In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier

# Load the dataset
df = pd.read_csv("heart_failure_data.csv")

# Merge 1 and 2 into a single class for binary classification
df['HF'] = df['HF'].apply(lambda x: 1 if x in [1, 2] else 0)

# Select features and target
df['EF_GLS']= df['EF']/ df['GLS']
X = df[['EF','GLS', 'QRS','EF_GLS']]
y = df['HF']

# Check class balance
print("Target class distribution:")
print(y.value_counts(normalize=True))

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

# Define parameter grid for GridSearchCV
param_grid = {
    'n_estimators': [50, 100, 150],
    'max_depth': [3, 4, 5],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1],
    'colsample_bytree': [0.8, 1],
}

# Initialize XGBoost model
xgb = XGBClassifier(eval_metric='logloss')

# GridSearchCV setup
grid_search = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    verbose=1,
    n_jobs=-1
)

# Fit the grid search
grid_search.fit(X_train, y_train)

# Best model
best_model = grid_search.best_estimator_
print("\nBest Parameters:", grid_search.best_params_)
print("Best Cross-Validated Accuracy: {:.4f}".format(grid_search.best_score_))

# Evaluate on test set
y_pred = best_model.predict(X_test)
y_pred_train = best_model.predict(X_train)

# Accuracy
accuracy_train = accuracy_score(y_train, y_pred_train)
accuracy_test = accuracy_score(y_test, y_pred)

# Error
error_train = 1 - accuracy_train
error_test = 1 - accuracy_test

# Evaluation
print("\nTraining Accuracy: {:.4f}".format(accuracy_train))
print("Testing Accuracy: {:.4f}".format(accuracy_test))
print("Training Error: {:.4f}".format(error_train))
print("Testing Error: {:.4f}".format(error_test))

# Classification report
print("\nClassification Report (Test Data):")
print(classification_report(y_test, y_pred))

# Cross-validation with best model
cv_scores = cross_val_score(best_model, X_scaled, y, cv=5)
print("\nCross-validated accuracy scores:", cv_scores)
print("Mean CV accuracy:", cv_scores.mean())

# Overfitting check
if accuracy_train == 1.0 and accuracy_test < 1.0:
    print("\nWarning: Overfitting detected!")
else:
    print("\nNo obvious overfitting detected.")

Target class distribution:
HF
0    0.5
1    0.5
Name: proportion, dtype: float64
Fitting 5 folds for each of 108 candidates, totalling 540 fits

Best Parameters: {'colsample_bytree': 0.8, 'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 50, 'subsample': 0.8}
Best Cross-Validated Accuracy: 0.9579

Training Accuracy: 0.9688
Testing Accuracy: 0.9167
Training Error: 0.0312
Testing Error: 0.0833

Classification Report (Test Data):
              precision    recall  f1-score   support

           0       0.92      0.92      0.92        12
           1       0.92      0.92      0.92        12

    accuracy                           0.92        24
   macro avg       0.92      0.92      0.92        24
weighted avg       0.92      0.92      0.92        24


Cross-validated accuracy scores: [0.91666667 1.         0.83333333 1.         0.95833333]
Mean CV accuracy: 0.9416666666666667

No obvious overfitting detected.
